# No Hack — Hybrid Stack Kaggle Training

This notebook runs only with **verified, canonical NoHackFlow v1 labeled CSV** files. It does **not** silently map CICFlowMeter, NSL-KDD or other incompatible benchmark columns. You must verify dataset usage, group/time leakage and permission before uploading public data. A Kaggle GPU may be enabled but the reference gradient-boosting implementation is CPU-compatible.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

DATASET_CSV = Path("/kaggle/input/YOUR_VERIFIED_NOHACKFLOW_DATASET/flows.csv")
WORKDIR = Path("/kaggle/working/nohack-source")
SPLITS = Path("/kaggle/working/nohack-splits")
MODEL = Path("/kaggle/working/nohack-model")
assert DATASET_CSV.is_file(), (
    f"Upload the verified canonical labeled dataset: {DATASET_CSV}"
)
print("Dataset CSV found:", DATASET_CSV.name, "bytes:", DATASET_CSV.stat().st_size)

## Install the code

Enable Kaggle notebook Internet access if permitted and needed to clone this repository. Review the GitHub branch and commit before execution. Do not use a random third-party model archive.

In [ ]:
subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        "feature/chrome-only-complete-stack",
        "https://github.com/shanmukhdatta/No-Hack-.git",
        str(WORKDIR),
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(WORKDIR) + "[ml-core]"],
    check=True,
)
sha = subprocess.check_output(
    ["git", "-C", str(WORKDIR), "rev-parse", "HEAD"], text=True
).strip()
print("No Hack training commit:", sha)

## Create frozen group-disjoint train/validation/test partitions

This step will refuse missing provenance groups, schema mismatch, label imbalance across splits and exact duplicate vectors crossing partitions.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(WORKDIR / "scripts" / "prepare_nohack.py"),
        "--input",
        str(DATASET_CSV),
        "--output",
        str(SPLITS),
        "--seed",
        "42",
    ],
    check=True,
)
manifest = json.loads((SPLITS / "manifest.json").read_text())
print("Rows per split:", {k: v["rows"] for k, v in manifest["splits"].items()})
print("Groups per split:", {k: v["groups"] for k, v in manifest["splits"].items()})

## Fit out-of-fold hybrid stack and evaluate held-out test

Training includes XGBoost, LightGBM, CatBoost, Isolation Forest, GMM, an OOF Logistic Regression stack and validation-only calibration. No results are prefilled.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(WORKDIR / "scripts" / "train_nohack.py"),
        "--data",
        str(SPLITS),
        "--output",
        str(MODEL),
        "--folds",
        "4",
        "--seed",
        "42",
    ],
    check=True,
)
report = json.loads((MODEL / "metrics.json").read_text())
print(json.dumps(report, indent=2))

## Review and package

Do not call results release-ready until PCAP label matching, test-set independence, false positives, attack-family taxonomy, local Windows feature parity and native Chrome host integration are verified.

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/nohack-model-outputs", "zip", root_dir=MODEL
)
print("Saved output bundle:", zip_path)
print("Bundle files:", [p.name for p in MODEL.iterdir()])
print(
    "Security: only load joblib you personally trained or verified from a trusted source."
)